# Belief lab

Every stage of the experiment behind sliders: generate the data, look at it, train
a model on it, snapshot it, probe the snapshots. Nothing here is hardcoded to the
four committed runs - change any physical constant, retrain in the same session,
and probe what comes out.

Run the setup cell, then the four tabs below in order. Each tab is independent
except that **Probe** needs checkpoints, which either come from **Train** in this
session or from the `experiments/outputs-messk` directory.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import ipywidgets as W
import numpy as np
from IPython.display import display

from models.explore import find_checkpoints, fit_readout, load_model, plot_sequence
from models.lab import Arch, counterfactual, probe_checkpoints, run_training
from models.labplots import plot_generation, plot_probe_table, plot_training
from models.train import pick_device
from physics.messk_configs import CHAIN, DRIVER, MESSK_CONFIGS, SYSTEMS, make_process

DEVICE = pick_device()
CHECKPOINT_DIR = Path("../experiments/outputs-messk")
SESSION_DIR = Path("../experiments/outputs-lab")
SESSION_DIR.mkdir(parents=True, exist_ok=True)

STATE = {"proc": None, "batch": None, "report": None, "rows": None}
print(f"device {DEVICE} | {len(find_checkpoints(CHECKPOINT_DIR))} saved checkpoints")

## 1. Simulator

The parameters are grouped by what they control. **Physics** is the system itself,
including the release condition and the damping. **Channel** is how the trajectory
becomes tokens. **Chain** is the hidden process the whole experiment is about.

`auto-fit range` is the one that needs explaining: `obs_range` decides where the
`n_obs` bins are laid down, and it is a fixed constant per system tuned for that
system's *default* parameters. Change the damping or the release and the
trajectory moves, so the bins no longer sit where the data is - you get clipping
at the edges, dead bins in the middle, or both. The button measures the
trajectory and re-lays the bins over it. The occupancy panel is where you check
whether it was needed.

In [ ]:
def _p(label, value, step=None, kind="float"):
    if kind == "int":
        return W.IntText(value=value, description=label, style={"description_width": "110px"},
                         layout=W.Layout(width="220px"))
    return W.FloatText(value=value, step=step, description=label,
                       style={"description_width": "110px"}, layout=W.Layout(width="220px"))

#: Per-system physical knobs beyond the shared ones. The values are read straight
#: off each dataclass so this list cannot drift from the code.
SYSTEM_FIELDS = {
    "pendulum":        ["g", "length", "gamma", "omega_max", "theta0", "omega0"],
    "predator_prey":   ["a", "b", "c", "d", "kappa", "x0", "y0"],
    "sphere":          ["g", "length", "gamma", "rate_max", "theta0", "psi_dot0"],
    "double_pendulum": ["g", "l1", "l2", "gamma1", "gamma2", "omega_max", "th1_0", "th2_0"],
}

sys_w      = W.Dropdown(options=sorted(MESSK_CONFIGS), value="pendulum_mess4", description="system")
delta_v_w  = _p("delta_v", SYSTEMS["pendulum"]["delta_v"], 0.05)
dt_w       = _p("dt", SYSTEMS["pendulum"]["dt"], 0.005)
n_steps_w  = W.IntSlider(value=DRIVER["n_steps"], min=2, max=30, description="physics ticks",
                         style={"description_width": "110px"})
m_w        = W.IntSlider(value=DRIVER["m"], min=4, max=40, description="chain ticks (m)",
                         style={"description_width": "110px"})
n_obs_w    = W.IntSlider(value=DRIVER["n_obs"], min=11, max=513, step=2, description="bins (n_obs)",
                         style={"description_width": "110px"})
lo_w, hi_w = _p("range lo", -1.5708, 0.05), _p("range hi", 1.5708, 0.05)
stay_w     = W.FloatSlider(value=CHAIN["stay"], min=0.25, max=0.99, step=0.01, description="stay",
                           style={"description_width": "110px"})
alpha_w    = W.FloatSlider(value=CHAIN["alpha"], min=0.25, max=0.99, step=0.01, description="alpha",
                           style={"description_width": "110px"})
nseq_w     = W.IntSlider(value=256, min=32, max=2048, step=32, description="sequences",
                         style={"description_width": "110px"})
seed_w     = _p("seed", 20260829, kind="int")
seq_w      = W.IntSlider(value=0, min=0, max=31, description="show sequence",
                         style={"description_width": "110px"})
tick_w     = W.IntSlider(value=3, min=0, max=15, description="branch at tick",
                         style={"description_width": "110px"})
autofit_b  = W.Button(description="auto-fit range", button_style="info")
gen_b      = W.Button(description="generate", button_style="primary")
gen_out    = W.Output()

physics_box = W.VBox([])

def _rebuild_physics(*_):
    name = MESSK_CONFIGS[sys_w.value]
    system = SYSTEMS[name]["factory"]()
    physics_box.children = [_p(f, getattr(system, f), 0.05) for f in SYSTEM_FIELDS[name]]
    delta_v_w.value, dt_w.value = SYSTEMS[name]["delta_v"], SYSTEMS[name]["dt"]
    lo_w.value, hi_w.value = system.obs_range
    m_w.value = DRIVER["m"]

sys_w.observe(_rebuild_physics, names="value")
_rebuild_physics()


def build_proc():
    overrides = {w.description: w.value for w in physics_box.children}
    overrides["obs_range"] = (lo_w.value, hi_w.value)
    return make_process(
        sys_w.value, system=overrides, chain={"stay": stay_w.value, "alpha": alpha_w.value},
        delta_v=delta_v_w.value, dt=dt_w.value, n_steps=n_steps_w.value,
        m=m_w.value, n_obs=n_obs_w.value,
    )


def _generate(_=None):
    proc = build_proc()
    batch = proc.sample_batch(np.random.default_rng(seed_w.value), nseq_w.value)
    STATE["proc"], STATE["batch"] = proc, batch
    seq_w.max, tick_w.max = nseq_w.value - 1, proc.m - 1
    report = proc.bin_report(batch["observable"])
    with gen_out:
        gen_out.clear_output(wait=True)
        print(f"{nseq_w.value} sequences x {proc.seq_len} tokens | "
              f"{report['used_bins']}/{report['n_obs']} bins used | "
              f"{report['clipped']:.2%} clipped | "
              f"observed {report['observed_range'][0]:+.3f}..{report['observed_range'][1]:+.3f} "
              f"vs range {report['obs_range'][0]:+.3f}..{report['obs_range'][1]:+.3f}")
        if report["clipped"] > 0.02:
            print(f"  clipping above 2% - press auto-fit range, or widen it by hand")
        cf = counterfactual(proc, batch["letters"][seq_w.value], tick_w.value)
        display(plot_generation(proc, batch, seq_w.value, tick_w.value, cf))


def _autofit(_=None):
    proc = build_proc()
    sample = proc.sample_batch(np.random.default_rng(seed_w.value), 128)
    lo_w.value, hi_w.value = proc.suggested_obs_range(sample["observable"])
    _generate()

gen_b.on_click(_generate)
autofit_b.on_click(_autofit)
for w in (seq_w, tick_w):
    w.observe(lambda c: _generate(), names="value")

sim_panel = W.VBox([
    W.HBox([
        W.VBox([W.HTML("<b>physics</b>"), sys_w, delta_v_w, dt_w, physics_box]),
        W.VBox([W.HTML("<b>channel</b>"), n_steps_w, m_w, n_obs_w, lo_w, hi_w, autofit_b]),
        W.VBox([W.HTML("<b>chain and sampling</b>"), stay_w, alpha_w, nseq_w, seed_w,
                seq_w, tick_w, gen_b]),
    ]),
    gen_out,
])

## 2. Train

Trains on whatever the Simulator tab last generated, so the model's vocabulary and
context are whatever you set there. Snapshots land in `experiments/outputs-lab/`
and feed the Probe tab directly.

Fraction `0.0` is the random initialisation, saved before the first step. Keep it:
it is the control that decides whether any probe number below means anything.

A few million tokens takes a minute or two here and is enough to see the loss
leave the uniform line. The committed runs used 500M and 1B - this tab is for
watching the shape, not for reproducing them.

In [ ]:
d_model_w = W.IntSlider(value=128, min=32, max=512, step=32, description="d_model",
                        style={"description_width": "110px"})
layers_w  = W.IntSlider(value=4, min=1, max=12, description="n_layers",
                        style={"description_width": "110px"})
heads_w   = W.IntSlider(value=1, min=1, max=8, description="n_heads",
                        style={"description_width": "110px"})
tokens_w  = W.FloatLogSlider(value=2e6, base=10, min=5, max=9, step=0.25, description="tokens",
                             style={"description_width": "110px"}, readout_format=".2g")
batch_w   = W.IntSlider(value=128, min=16, max=512, step=16, description="batch",
                        style={"description_width": "110px"})
lr_w      = W.FloatLogSlider(value=1e-3, base=10, min=-5, max=-2, step=0.25, description="lr",
                             style={"description_width": "110px"}, readout_format=".1e")
frac_w    = W.Text(value="0.0,0.1,0.25,0.5,1.0", description="snapshot at",
                   style={"description_width": "110px"})
runname_w = W.Text(value="lab", description="run name", style={"description_width": "110px"})
train_b   = W.Button(description="train", button_style="primary")
train_out = W.Output()

def _train(_=None):
    if STATE["proc"] is None:
        with train_out:
            print("generate data on the Simulator tab first")
        return
    proc = STATE["proc"]
    with train_out:
        train_out.clear_output(wait=True)
        arch = Arch(d_model=d_model_w.value, n_layers=layers_w.value, n_heads=heads_w.value)
        started = time.perf_counter()
        report = run_training(
            proc, arch, SESSION_DIR, name=runname_w.value,
            total_tokens=int(tokens_w.value), batch_size=batch_w.value,
            learning_rate=lr_w.value,
            fractions=tuple(float(f) for f in frac_w.value.split(",")),
            device=DEVICE,
        )
        report["batch_size"] = batch_w.value
        STATE["report"] = report
        print(f"{report['steps']} steps in {time.perf_counter() - started:.0f}s | "
              f"final eval {report['final']['eval_loss']:.3f} nats "
              f"against uniform {report['uniform_loss']:.3f} | "
              f"{len(report['checkpoint_paths'])} snapshots in {SESSION_DIR}")
        display(plot_training(report))

train_b.on_click(_train)
train_panel = W.VBox([
    W.HBox([W.VBox([W.HTML("<b>architecture</b>"), d_model_w, layers_w, heads_w]),
            W.VBox([W.HTML("<b>optimisation</b>"), tokens_w, batch_w, lr_w]),
            W.VBox([W.HTML("<b>run</b>"), frac_w, runname_w, train_b])]),
    train_out,
])

## 3. Probe

Fits a ridge readout from the residual stream to the belief and the metric, and
scores it on sequences the fit never saw.

**Read the two bars against each other, not on their own.** The metric bar is high
even at snapshot `0`, because the physical state is close to a linear function of
the tokens in the window and an untrained network's random projection preserves
that. The belief bar at snapshot `0` is the number any later belief bar has to
beat. A belief bar that merely looks high is not a result.

`whole-tick` concatenates all `n_steps` positions of a tick; turning it off keeps
only the last. That choice moves the metric score more than training does, which
is worth seeing once directly.

In [ ]:
source_w   = W.RadioButtons(options=["this session", "saved runs"], description="checkpoints",
                            style={"description_width": "110px"})
saved_w    = W.SelectMultiple(options=sorted(find_checkpoints(CHECKPOINT_DIR)), rows=8,
                              description="files", style={"description_width": "110px"},
                              layout=W.Layout(width="520px"))
depths_w   = W.IntSlider(value=4, min=1, max=9, description="depths",
                         style={"description_width": "110px"})
whole_w    = W.Checkbox(value=True, description="whole-tick readout")
ridge_w    = W.FloatLogSlider(value=100.0, base=10, min=-1, max=5, description="ridge alpha",
                              style={"description_width": "110px"})
nfit_w     = W.IntSlider(value=3072, min=512, max=8192, step=512, description="fit seqs",
                         style={"description_width": "110px"})
probe_b    = W.Button(description="probe", button_style="primary")
probe_out  = W.Output()

def _train_probe(_=None):
    if STATE["proc"] is None:
        with probe_out:
            print("generate data on the Simulator tab first")
        return
    if source_w.value == "this session":
        if STATE["report"] is None:
            with probe_out:
                print("no run in this session - train one, or switch to saved runs")
            return
        paths = STATE["report"]["checkpoint_paths"]
    else:
        found = find_checkpoints(CHECKPOINT_DIR)
        paths = {name: str(found[name]) for name in saved_w.value}
        if not paths:
            with probe_out:
                print("select one or more files")
            return
    with probe_out:
        probe_out.clear_output(wait=True)
        rows = probe_checkpoints(
            STATE["proc"], paths, device=DEVICE, n_fit=nfit_w.value,
            n_depths=depths_w.value, whole_tick=whole_w.value, alpha=ridge_w.value,
        )
        STATE["rows"] = rows
        for r in rows:
            print(f"{str(r['label']):>42s}  belief {r['belief_r2']:+.3f}  "
                  f"metric {r['metric_r2']:+.3f}  loss {r['eval_loss']:.3f}  "
                  f"({r['rows_per_feature']:.1f} rows/feature)")
        display(plot_probe_table(rows))

probe_b.on_click(_train_probe)
probe_panel = W.VBox([
    W.HBox([W.VBox([W.HTML("<b>which models</b>"), source_w, saved_w]),
            W.VBox([W.HTML("<b>readout</b>"), depths_w, whole_w, ridge_w, nfit_w, probe_b])]),
    probe_out,
])

## 4. Truth against model

One sequence, one checkpoint: the exact belief and metric against what a linear
readout recovers, plus the model's own next-token distribution, which has no probe
between it and the picture.

In [ ]:
ckpt_w    = W.Dropdown(options=sorted(find_checkpoints(CHECKPOINT_DIR)) or ["<none>"],
                       description="checkpoint", style={"description_width": "110px"},
                       layout=W.Layout(width="480px"))
refresh_b = W.Button(description="refresh list")
compare_w = W.IntSlider(value=0, min=0, max=31, description="sequence",
                        style={"description_width": "110px"})
compare_b = W.Button(description="show", button_style="primary")
compare_out = W.Output()

def _refresh(_=None):
    found = find_checkpoints(CHECKPOINT_DIR) | find_checkpoints(SESSION_DIR)
    ckpt_w.options = sorted(found) or ["<none>"]
    STATE["found"] = found

_refresh()

def _compare(_=None):
    if STATE["proc"] is None:
        with compare_out:
            print("generate data on the Simulator tab first")
        return
    proc, batch = STATE["proc"], STATE["batch"]
    with compare_out:
        compare_out.clear_output(wait=True)
        model = load_model(STATE["found"][ckpt_w.value], device=DEVICE)
        if proc.seq_len > model.config.n_ctx or proc.n_obs > model.config.vocab_size:
            print(f"this checkpoint wants n_ctx {model.config.n_ctx} and vocab "
                  f"{model.config.vocab_size}; the current data is {proc.seq_len} x {proc.n_obs}")
            return
        readout = fit_readout(model, proc, device=DEVICE, n_fit=nfit_w.value,
                              n_depths=min(depths_w.value, model.config.n_layers + 1),
                              whole_tick=whole_w.value, alpha=ridge_w.value)
        print(f"{model.config.n_layers} layers, d_model {model.config.d_model} | "
              f"belief {readout.r2['belief']:+.3f}  metric {readout.r2['metric']:+.3f} "
              f"({readout.rows_per_feature:.1f} rows/feature)")
        display(plot_sequence(proc, batch, min(compare_w.value, len(batch["tokens"]) - 1),
                              model, readout, DEVICE))

refresh_b.on_click(_refresh)
compare_b.on_click(_compare)
compare_panel = W.VBox([W.HBox([ckpt_w, refresh_b, compare_w, compare_b]), compare_out])

In [ ]:
tabs = W.Tab(children=[sim_panel, train_panel, probe_panel, compare_panel])
for i, title in enumerate(["1. Simulator", "2. Train", "3. Probe", "4. Truth vs model"]):
    tabs.set_title(i, title)
display(tabs)
_generate()

## Things worth doing

- **Turn the damping to zero** on the Simulator tab. The trajectory stops decaying,
  the observable spends its time near the turning points, and the occupancy panel
  shows the bins piling up at the edges.
- **Drop `n_obs` to 21.** The counterfactual panel is the one to watch: the four
  alternate futures start landing in the same bin, so the letters stop being
  distinguishable from the observations, and the belief becomes unrecoverable in
  principle rather than just in practice.
- **Raise `delta_v` until the occupancy panel reports clipping.** Clipped samples
  lose their value, so the kick becomes *more* visible for the wrong reason.
- **Train two models at the same budget, `d_model` 64 and 256, and probe both.**
  The belief bars separate; the metric bars do not.
- **Probe snapshot `0` against the final one.** The metric bar barely moves. That
  gap between the two bars is the actual result of this experiment.